# FEniCS showcase: seven real dolfinx studies as composable pbg processes

_Investigation `fenics-showcase` — coder reproduction notebook._

**Question.** Can real FEniCSx (dolfinx) solvers, wrapped as ordinary process-bigraph
Steps/Processes, both PASS rigorous numerical-verification checks and
drive seven genuinely different, showcase-tier physics problems --
spanning high-order accuracy, singular geometry, morphogen patterning,
Turing instability, bluff-body vortex shedding, porous-media flow, and
moving-boundary fluid-structure coupling -- using the same underlying
bridge throughout?

A showcase investigation proving that a real external FEM solver
(FEniCSx/dolfinx, not a mock or reimplementation), wrapped as ordinary
process-bigraph Steps/Processes, both passes rigorous numerical-
verification checks and drives seven substantially different
showcase-tier physics problems using the same bridge throughout. Two
validation studies (`poisson-validation`, `mesh-convergence`) confirm
numerical fidelity -- optimal multi-order L2 convergence and adaptive
recovery of an optimal rate on a corner singularity. Two dynamics
studies (`transient-diffusion`, `reaction-diffusion`) extend the bridge
to time-stepping and cross-process store-wiring composability, the
latter producing a genuine Turing instability from three
independently-authored processes. Three ADVANCED studies
(`navier-stokes`, `complex-geometry`, `moving-boundary`) push the same
bridge into incompressible Navier-Stokes vortex shedding, a real
porous-media permeability result, and ALE fluid-structure-coupled
peristaltic pumping.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-fenics/viva-fenics').is_dir():
    REPO = Path('/home/runner/work/viva-fenics/viva-fenics')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_fenics.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

## Study: `poisson-validation`

**Question.** Do P1, P2, and P3 Lagrange elements each achieve their OWN theoretically
optimal L2 convergence rate (degree+1: 2, 3, 4 respectively) when solving
the dolfinx-backed `PoissonSolverStep` on a smooth (C-infinity)
manufactured solution, or does any element order under- or over-perform
its textbook rate?

**Objective.** For degree in {1, 2, 3} and a resolution sweep (mesh cells per side: 8,
16, 32, 64), solve -div(grad(u)) = f on the unit square with f and the
Dirichlet BC derived from u_exact = sin(pi*x)*sin(pi*y), compute each
point's L2 error against the true exact solution, and fit the log-log
convergence rate (slope of log error vs log h) per degree. Confirm P1's
rate lands near 2.0, P2's near 3.0, and P3's near 4.0 -- a rigorous
multi-order verification, not a single "error is small" checkbox.

**Hypothesis.** u_exact = sin(pi*x)*sin(pi*y) is smooth but NOT polynomial, so unlike a
quadratic manufactured solution (exactly representable by degree>=2
elements, collapsing their error to round-off) no fixed-degree Lagrange
space represents it exactly. Standard a priori FEM theory predicts every
degree-p Lagrange element converges in L2 at rate p+1 on a sufficiently
smooth problem: P1 -> O(h^2), P2 -> O(h^3), P3 -> O(h^4). A genuine
dolfinx solve, with the L2 error measured against the true symbolic exact
solution at elevated quadrature (not an FE interpolant of it, which would
understate exactly the higher-order error this check needs to see),
should reproduce all three optimal rates cleanly.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: poisson-validation ===
STUDY = 'poisson-validation'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**convergence-multi-order**


In [ ]:
# convergence-multi-order
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**solution-heatmap**


In [ ]:
# solution-heatmap
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| p1-achieves-optimal-rate | kind=derived_scalar field=p1_rate | op range low 1.75 high 2.25 provenance {'kind': 'theory', 'note': 'A priori FEM theory: degree-p Lagrange elements converge in L2 at O(h^(p+1)) on a sufficiently smooth problem; p=1 -> rate 2.0. 0.25 is a generous margin around that theoretical value, not a fitted threshold; the achieved production rate is 1.998.'} |
| p2-achieves-optimal-rate | kind=derived_scalar field=p2_rate | op range low 2.75 high 3.25 provenance {'kind': 'theory', 'note': 'p=2 -> theoretical L2 rate 3.0. 0.25 margin around theory; achieved production rate is 3.000.'} |
| p3-achieves-optimal-rate | kind=derived_scalar field=p3_rate | op range low 3.75 high 4.25 provenance {'kind': 'theory', 'note': "p=3 -> theoretical L2 rate 4.0, the most demanding of the three orders (requires elevated quadrature so the check isn't itself quadrature-limited -- see fem.l2_error_exact). 0.25 margin around theory; achieved production rate is 4.000."} |


## Study: `mesh-convergence`

**Question.** On the classic L-shaped-domain re-entrant-corner Laplace singularity --
where the exact solution's gradient blows up at the corner and UNIFORM
refinement is provably capped at a suboptimal convergence rate -- does a
real residual-based a posteriori error estimator, driving Doerfler-marked
ADAPTIVE refinement, concentrate elements at the corner and recover
near-optimal convergence?

**Objective.** Run a uniform-refinement sequence and a residual-based adaptive (AMR)
sequence from the same coarse initial L-shaped mesh, fit the energy-norm-
error-vs-DOFs log-log slope for each, and confirm (a) the adaptive slope
is substantially steeper (closer to the optimal -1/2) than the uniform
slope (capped near -1/3), and (b) the adaptive mesh's cell density near
the re-entrant corner, relative to the far field, grows dramatically
relative to the initial mesh -- direct geometric evidence the estimator
is doing its job, not just producing a better number by coincidence.

**Hypothesis.** The L-shaped domain's re-entrant corner (interior angle 3*pi/2) has exact
harmonic solution u = r^(2/3)*sin(2*theta/3), whose H^(5/3-eps) regularity
caps P1 UNIFORM refinement's energy-norm convergence at O(dofs^-1/3)
instead of the usual O(dofs^-1/2). A residual-based edge-jump error
estimator should produce indicators that spike sharply at the corner
(where u_h's piecewise-linear gradient most disagrees with its
neighbors), so Doerfler (bulk-chasing) marking on those indicators should
concentrate new elements there and recover close to the optimal
O(dofs^-1/2) rate despite the singularity.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mesh-convergence ===
STUDY = 'mesh-convergence'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**mesh-refinement-animation**


In [ ]:
# mesh-refinement-animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**convergence-comparison**


In [ ]:
# convergence-comparison
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**final-solution-heatmap**


In [ ]:
# final-solution-heatmap
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| adaptive-beats-uniform-rate | kind=derived_scalar field=adaptive_rate | op <= value -0.45 provenance {'kind': 'theory', 'note': "Standard a priori FEM theory for a corner singularity of exponent 2/3 caps P1 uniform refinement's energy-norm rate near O(dofs^-1/3) (theoretical -0.333); optimally graded/adaptive refinement recovers close to the smooth-solution rate O(dofs^-1/2). -0.45 is a calibration margin above (i.e. less steep than) the theoretical -0.5, not a fitted threshold; the achieved production rate is -0.528."} |
| refinement-concentrates-at-corner | kind=derived_scalar field=corner_density_ratio | op >= value 15.0 provenance {'kind': 'calibration', 'note': 'The initial (near-uniform, pre-refinement) mesh\'s own near/far density ratio is 0.52 -- the natural "no bias" baseline for this domain/radii choice. The achieved production adaptive mesh\'s ratio is 29.17 (55.8x growth). 15.0 sits with large headroom below that achieved value while still failing a scheme that spread refinement uniformly (which would keep the ratio near the initial ~0.5).'} |


## Study: `transient-diffusion`

**Question.** Do two independently-authored processes -- ``DiffusionProcess`` (extended
with a real Dirichlet source boundary c=c0 at x=0) and
``LinearDegradationProcess`` (the pure-numpy rate ``-k*c``), coupled ONLY
through shared bigraph stores -- produce a genuine exponential morphogen
gradient c(x) = c0*exp(-x/lambda) at steady state, with decay length
lambda = sqrt(D/k) matching the classic Source-Diffusion-Degradation (SDD)
analytic prediction, when driven through process-bigraph's own Composite
tick loop? And do Wolpert "French flag" positional-information thresholds
applied to that gradient partition the domain into 3 fate regions whose
boundaries shift predictably as lambda shrinks?

**Objective.** Run the ``morphogen_gradient`` composite to steady state across THREE
decay-length regimes (D fixed, k swept: 1.0/2.0/4.0, giving
lambda=0.316/0.224/0.158) and confirm (a) each regime's steady profile is
genuinely exponential -- a fitted decay length within 10% of the analytic
lambda=sqrt(D/k), not just a qualitatively decreasing field -- (b) fields
stay bounded in [0, c0] (no accumulate-vs-overwrite blow-up, no unphysical
negative concentration), and (c) French-flag threshold crossings
(c=0.5*c0, c=0.1*c0) land within 10% of their analytic x-positions AND
shift monotonically inward as k increases -- demonstrating lambda's
control over the positional-information map is a real, tunable property
of the composed system, not a fixed artifact of one parameter set.

**Hypothesis.** Neither process implements SDD itself: DiffusionProcess only knows how to
solve backward-Euler diffusion with an optional fixed-value boundary, and
LinearDegradationProcess only knows how to read a field and write back
``-k*c``. If the shared-store wiring genuinely couples them, a field
seeded at zero should build up from the source boundary into a steady
gradient whose SHAPE is exponential (a straight line on a log-linear plot)
and whose fitted decay length matches lambda=sqrt(D/k) -- not just "some
decreasing curve" -- and sweeping k at fixed D should shrink that decay
length in the analytically predicted sqrt(D/k) proportion, moving the
French-flag threshold crossings inward in lockstep.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: transient-diffusion ===
STUDY = 'transient-diffusion'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**morphogen-animation**


In [ ]:
# morphogen-animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**morphogen-profile-fit**


In [ ]:
# morphogen-profile-fit
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**morphogen-french-flag-baseline**


In [ ]:
# morphogen-french-flag-baseline
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**morphogen-french-flag-short**


In [ ]:
# morphogen-french-flag-short
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**morphogen-french-flag-shorter**


In [ ]:
# morphogen-french-flag-shorter
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| exponential-gradient | kind=derived_scalar field=lambda_rel_err_max | op < value 0.1 provenance {'kind': 'calibration', 'note': 'The committed production run (resolution=64, 800 ticks/regime) achieved max rel_err=1.63% (baseline k=1.0; short-range/shorter- range regimes were 0.23%/0.01%). 10% leaves >6x headroom below that while still catching a genuinely disconnected source-boundary or degradation term (which would produce a non-exponential or wrong-slope profile, not a small numerical discrepancy).'} |
| fields-bounded-positive | kind=derived_scalar field=fields_bounded_positive | op == value True provenance {'kind': 'theory', 'note': "A degrading, boundary-sourced field with no other input can never physically exceed its boundary value c0 or go negative; this is also the field-range guard against the accumulate-vs-overwrite regression class (stores.source composing additively instead of being overwritten each tick), which would blow the field well past this band -- see this study's report for the achieved c_range=[0.0036, 1.0000] (shorter-range) through [0.0845, 1.0000] (baseline)."} |
| french-flag-thresholds | kind=derived_scalar field=boundary_shift_baseline_minus_shorter | op >= value 0.05 provenance {'kind': 'calibration', 'note': 'The committed production run achieved x_high rel_err <=0.8% across all 3 regimes and a baseline-minus-shorter-range x_high shift of 0.111 (0.221 -> 0.110), with x_low shifting 0.811 -> 0.364 over the same sweep. 0.05 leaves >2x headroom below the achieved shift while still catching a "lambda has no real effect" regression.'} |


## Study: `reaction-diffusion`

**Question.** Do three independently-authored processes -- TWO ``DiffusionProcess``
instances (one per species, U at Du and V at Dv=Du/2, each with no
knowledge of reactions) and ONE ``GrayScottReactionProcess`` (no knowledge
of FEM/diffusion) -- produce a genuine 2D Turing instability when wired to
the same bigraph stores: does a near-uniform field spontaneously
self-organize into spots/stripes/labyrinth patterns, with none of the
three processes calling into either of the others?

**Objective.** Run the ``turing_patterns`` composite (2x DiffusionProcess ⊕
GrayScottReactionProcess) across three Gray-Scott parameter regimes and
confirm (a) the baseline regime's V field grows substantially more
spatially heterogeneous over the run (pattern EMERGES from a near-uniform
start), (b) the resulting pattern is bounded and structurally
non-trivial -- a finite fraction of the domain organizes into pattern
features, not the whole domain and not none of it -- and (c) different
(F, k) regimes produce measurably different pattern morphology (coverage
fraction), demonstrating the composed system is a genuine, tunable Turing
system, not a fixed numerical artifact.

**Hypothesis.** Coupling is entirely a property of the document wiring (shared
``stores.u`` / ``stores.v`` / ``stores.source_u`` / ``stores.source_v``
paths), not of any one process's implementation: starting from U~1, V~0
everywhere plus a small seeded perturbation, the differential diffusion
(Du > Dv) between the two DiffusionProcess instances, combined with the
GrayScottReactionProcess's nonlinear feed/kill kinetics, should amplify
the perturbation into a spatially heterogeneous, bounded pattern -- not
decay back to uniformity, and not blow up -- and different (F, k) kinetic
parameters should produce visibly different pattern morphologies.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: reaction-diffusion ===
STUDY = 'reaction-diffusion'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**turing-pattern-animation**


In [ ]:
# turing-pattern-animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**turing-pattern-baseline-final**


In [ ]:
# turing-pattern-baseline-final
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**turing-pattern-labyrinth-final**


In [ ]:
# turing-pattern-labyrinth-final
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**turing-pattern-stripes-final**


In [ ]:
# turing-pattern-stripes-final
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| pattern-emerges | kind=derived_scalar field=growth_ratio | op >= value 5.0 provenance {'kind': 'calibration', 'note': 'The committed production run (resolution=96, 6000 ticks, dt=1.0) achieved growth=10.5x (var(V) 0.00111 -> 0.01169); 5.0 leaves ~2.1x headroom below that while still catching a "reaction never actually coupled in" regression (which produces order-1 growth from quadrature noise alone, per the analogous check in ``reaction_diffusion``\'s Fisher-KPP study).'} |
| pattern-structured-not-blowup | kind=derived_scalar field=coverage_fraction | op in_range low 0.1 high 0.7 provenance {'kind': 'calibration', 'note': 'The committed production run\'s baseline achieved coverage=0.483 (V in [0.004, 0.367], comfortably bounded); [0.1, 0.7] brackets that with headroom on both sides while still catching "pattern never really emerged" (near 0) or "whole domain saturated/blew up" (near 1). The field-bound half of this check mirrors the FAST unit tests\' (-0.5, 1.5) band, itself the accumulate-vs-overwrite regression guard (see ``tests/test_turing_patterns.py``).'} |
| regimes-differ | kind=derived_scalar field=coverage_spread | op >= value 0.1 provenance {'kind': 'calibration', 'note': 'The committed production run achieved coverage 0.483 (baseline) / 0.290 (labyrinth) / 0.148 (stripes) -- a spread of 0.336. 0.1 leaves ~3.4x headroom below that while still catching a "regimes don\'t actually differ" regression. Coverage is partly confounded by the variants\' smaller step budget/coarser mesh (an explicit, disclosed wall-time tradeoff -- see the study report); the labyrinth regime\'s connected-component count (9, vs 1 for the other two regimes) is a second, less-confounded topological signal of the same claim, not gated on here only because it isn\'t defined identically enough across regimes for a clean numeric threshold.'} |


## Study: `navier-stokes`

**Question.** Does a real dolfinx incompressible Navier-Stokes solve -- Taylor-Hood-ish
P2/P1 velocity/pressure, classic IPCS operator splitting, on a
gmsh-generated channel-with-cylinder mesh -- reproduce the DFG 2D-2
benchmark's von Karman vortex street: periodic, self-sustaining vortex
shedding off an off-center cylinder, with drag/lift coefficients and a
Strouhal number in the neighborhood of the published reference values?

**Objective.** Run the `vortex_street` composite (real dolfinx IPCS on a gmsh-generated,
cylinder-refined channel mesh) long enough to pass through the impulsive-
start transient into established periodic shedding, and verify (a) the
lift coefficient genuinely oscillates (not just noisy/flat), (b) the mean
drag coefficient sits in the neighborhood of the DFG 2D-2 benchmark's
Cd_max~=3.22-3.24, and (c) the shedding frequency's Strouhal number sits
in the neighborhood of the benchmark's St~=0.30.

**Hypothesis.** At Re=100 with the cylinder offset 0.005 above the channel centerline (the
DFG 2D-2 geometry), the wake behind the cylinder is linearly unstable: an
impulsively-started, initially near-symmetric flow should grow a small
asymmetric perturbation (seeded by the geometric offset and numerical
round-off) into a fully periodic, alternating vortex street, visible
directly as an oscillating lift coefficient (mean ~0, non-zero amplitude)
and an oscillating drag coefficient (positive mean, smaller-amplitude
ripple at twice the shedding frequency) once the flow leaves the startup
transient. The oscillation frequency, non-dimensionalized as a Strouhal
number St = f*D/U, should land in the neighborhood of the DFG reference
~0.30 even if this solver's mesh/dt (chosen for feasible wall-time, not
full DFG-grade accuracy) don't reproduce the reference Cd_max/Cl_max to
the last decimal.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: navier-stokes ===
STUDY = 'navier-stokes'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vortex-street-vorticity**


In [ ]:
# vortex-street-vorticity
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**drag-lift-timeseries**


In [ ]:
# drag-lift-timeseries
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**wake-velocity-snapshot**


In [ ]:
# wake-velocity-snapshot
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**pressure-snapshot**


In [ ]:
# pressure-snapshot
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| lift-oscillates | kind=derived_scalar field=cl_std_analysis_window | op > value 0.15 provenance {'kind': 'calibration', 'note': '0.15 sits well above the ~1e-3 lift-coefficient noise floor observed during the ramp-phase transient in development, and comfortably below the production run\'s achieved std(Cl)=0.41 (analysis window t=1.5-3.0s), so it cleanly separates "no shedding" from the real shedding this run produced.'} |
| drag-in-benchmark-range | kind=derived_scalar field=cd_mean_analysis_window | op in_range low 2.7 high 3.6 provenance {'kind': 'calibration', 'note': "Schafer/Turek DFG 2D-2 reference Cd_max~=3.22-3.24 (unsteady case, Re=100) is the target this band is drawn near, but the band itself was fit around the production run's achieved Cd_mean=3.03 / Cd_max=3.15 (~3-6% below reference) on this solver's feasible-wall-time mesh/dt, not DFG-grade mesh convergence -- i.e. calibrated to the observed result with headroom, not derived from theory alone."} |
| strouhal-in-range | kind=derived_scalar field=strouhal_number | op in_range low 0.25 high 0.4 provenance {'kind': 'calibration', 'note': "Schafer/Turek DFG 2D-2 reference St~=0.30 (unsteady case, Re=100) is the target this band is drawn near, but the band itself was fit around the production run's achieved St~=0.31 +/- 0.07 (the uncertainty is the FFT's own bin-resolution floor over the ~1.5s/ ~5-cycle analysis window, not an error bar on the underlying physics) -- i.e. calibrated to the observed result with headroom, not derived from theory alone."} |


## Study: `complex-geometry`

**Question.** Does a real gmsh-constructed periodic array of circular pillars --
imported into dolfinx via `dolfinx.io.gmsh.model_to_mesh` -- support a
genuine steady Stokes (creeping) flow solve, driven by a prescribed
pressure drop, that stays approximately divergence-free and exactly
no-slip on every pillar and channel wall, and does the effective (Darcy)
permeability computed from that flow DECREASE monotonically as the
lattice's porosity decreases -- the Kozeny-Carman-type physical trend
expected of any real porous microstructure?

**Objective.** Build the `porous_lattice` composite (a single `PorousFlowStep`: gmsh
pillar-lattice geometry construction -> dolfinx Taylor-Hood import ->
steady Stokes solve -> Darcy permeability) for a baseline porosity plus 3
porosity variants (sweeping `pillar_radius` at fixed 4x4 lattice density)
and verify (a) every solve is approximately divergence-free and exactly
no-slip on every pillar/wall boundary, and (b) the effective permeability
k_eff decreases monotonically as porosity decreases across all 4 points.

**Hypothesis.** A real Taylor-Hood (P2 velocity / P1 pressure) mixed dolfinx solve on the
gmsh-generated pillar-lattice mesh, driven by a "do-nothing"
pressure-boundary-load formulation (no-slip on walls+pillars, pressure
prescribed only as a natural momentum-equation load at the open
inflow/outflow boundaries -- see `viva_fenics/processes/flow.py`'s module
comment above `PorousFlowStep`), should produce a physically valid
creeping-flow field on every lattice density tried, and the
volume-averaged x-velocity this field produces -- fed through Darcy's law
-- should fall as the pillar packing gets denser (porosity drops),
matching the qualitative Kozeny-Carman scaling (k ~ phi^3/(1-phi)^2) even
though a 4-point sweep cannot independently validate that scaling's exact
exponents.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: complex-geometry ===
STUDY = 'complex-geometry'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**flow-field**


In [ ]:
# flow-field
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**velocity-magnitude**


In [ ]:
# velocity-magnitude
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**permeability-vs-porosity**


In [ ]:
# permeability-vs-porosity
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| mesh-imports-with-tagged-boundaries | kind=derived_scalar field=min_n_cells_across_variants | op > value 20 provenance {'kind': 'calibration', 'note': "Observed cell counts at nx=ny=4, h_pillar=0.015 run 5000-9000 across the porosity sweep (see studies/complex-geometry/sims/run.py's printed summary); 20 is a generous floor that only fails on a genuinely degenerate/empty import."} |
| stokes-flow-divergence-free-and-noslip | kind=derived_scalar field=max_divergence_mean_across_variants | op < value 0.01 provenance {'kind': 'calibration', 'note': "Observed mean|div(u)| at production resolution runs 3e-5-1e-4 across the porosity sweep (a real Taylor-Hood LBB-stable pairing is much closer to exact incompressibility than this investigation's IPCS processes) -- 1e-2 leaves two orders of magnitude of headroom while still catching a genuinely broken assembly."} |
| noslip-satisfied-on-pillars | kind=derived_scalar field=max_noslip_speed_across_variants | op < value 1e-08 provenance {'kind': 'theory', 'note': 'Dirichlet BC dof elimination zeroes these dofs to machine precision by construction; observed max is exactly 0.0 in development runs, a 1e-8 floor leaves comfortable headroom for solver round-off while still catching a genuinely leaked/unapplied BC.'} |
| permeability-decreases-with-porosity | kind=derived_scalar field=min_consecutive_k_eff_ratio | op > value 1.0 provenance {'kind': 'theory', 'note': "Darcy's law at fixed driving pressure drop: a denser pillar packing leaves less open pore space, so the volume-averaged velocity (and hence k_eff) must fall as porosity falls -- see this study's expected_behavior for the full argument. The observed 4-point sweep shows a >10x drop in k_eff from the sparsest to the densest lattice (see studies/complex-geometry/sims/run.py's printed summary for the exact achieved values)."} |


## Study: `moving-boundary`

**Question.** Does composing a mesh-motion process (a real harmonic-extension ALE
solve, prescribing a traveling-wave occlusion on a channel's walls) with
a flow process (real IPCS incompressible Navier-Stokes, ALE-corrected for
the moving mesh), coupled ONLY through shared bigraph stores, reproduce
genuine peristaltic pumping -- a net axial flow rate that is positive
(in the wave's direction) and grows with the occlusion amplitude -- from
a real dolfinx solve, with no term in either process hard-coding the
other's physics?

**Objective.** Run the `peristalsis` composite (`PeristalticWallProcess` ⊕
`PeristalticFlowProcess`, coupled through `mesh_displacement_y` /
`mesh_velocity_y` / `wall_time` shared stores -- see
`viva_fenics/processes/peristalsis.py`'s module docstring) at three
occlusion amplitudes and verify (a) the baseline's post-transient,
time-averaged net flow rate Q (`mean_ux`, the domain-averaged
x-velocity) is genuinely positive, and (b) Q increases monotonically
across the amplitude sweep.

**Hypothesis.** A traveling constriction squeezes fluid forward faster than it can
relax backward (the textbook peristalsis mechanism), so the domain-
averaged x-velocity, time-averaged after the startup transient, should
be measurably POSITIVE for a nonzero occlusion amplitude and should
increase as the occlusion amplitude increases (a larger constriction
displaces proportionally more fluid per wave period) -- entirely an
emergent property of the composition (`PeristalticWallProcess`'s
mesh-motion + `PeristalticFlowProcess`'s ALE-Navier-Stokes), since
neither process alone computes a flow rate or knows the other exists.


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: moving-boundary ===
STUDY = 'moving-boundary'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**peristalsis-animation**


In [ ]:
# peristalsis-animation
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**net-flow-vs-amplitude**


In [ ]:
# net-flow-vs-amplitude
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| net-flow-is-positive | kind=derived_scalar field=net_flow_rate_baseline | op > value 0.02 provenance {'kind': 'calibration', 'note': "0.02 sits well below the achieved baseline Q~=0.19-0.2 (production run, this solver's feasible-wall-time mesh/dt), leaving generous headroom while still requiring genuine, clearly-nonzero pumping."} |
| flow-grows-with-amplitude | kind=derived_scalar field=net_flow_rate_monotonic | op == value True provenance {'kind': 'theory', 'note': "A larger occlusion amplitude displaces proportionally more fluid per wave period; the monotonic ordering is the qualitative peristaltic-transport signature this study validates (an exact pumping-rate formula is not required -- see the study's description)."} |
| wall-motion-genuinely-deforms-domain | kind=derived_scalar field=min_domain_area_fraction | op < value 0.92 provenance {'kind': 'theory', 'note': "domain_area is a real FEM-assembled integral over the current mesh; since L spans an integer number of full wavelengths, domain_area(t) is analytically constant at L*(H-amplitude) = 0.85*L*H for the baseline's amplitude=0.3 (the traveling wave's spatial cos-term integrates to exactly 0 over a whole wavelength). 0.92 leaves comfortable headroom below that exact, achieved value while still failing a near-1.0 (undeformed/un-coupled) reading, so a near-undeformed domain_area would indicate the mesh-motion -> flow coupling is not actually wired."} |
